
# Análise de Negócio — IMDb Movies

**Objetivo:** investigar, com Python e Pandas, o que parece influenciar a aceitação do público e como ela se relaciona com o retorno financeiro.

Este notebook responde às **9 perguntas de negócio** propostas no material da atividade. A análise separa:
- evidência encontrada nos dados;
- interpretação de negócio;
- limitações do dataset.

> **Arquivo esperado:** `imdb_movies.csv` na mesma pasta deste notebook.



## Contexto da análise

O cliente quer entender dois pontos antes de decidir onde investir:

1. o que faz um filme ser bem aceito pelo público;
2. o que faz um filme gerar retorno financeiro;

e, principalmente, **onde essas duas dimensões não caminham juntas**.

A análise não pretende provar causalidade. Correlação e diferenças entre grupos serão usadas como evidências para levantar hipóteses de negócio.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

ARQUIVO = "imdb_movies.csv"

df = pd.read_csv(ARQUIVO)

print(f"Linhas: {df.shape[0]:,}")
print(f"Colunas: {df.shape[1]}")
display(df.head())


In [ ]:

# Padronização básica
df.columns = df.columns.str.strip()

for col in ["score", "budget_x", "revenue"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["names", "genre", "crew", "orig_title", "status", "orig_lang", "country"]:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

# Datas
df["date"] = pd.to_datetime(df["date_x"].astype("string").str.strip(),
                            errors="coerce", format="%m/%d/%Y")
df["year"] = df["date"].dt.year

# Variáveis derivadas
df["log_budget"] = np.log10(df["budget_x"].where(df["budget_x"] > 0))
df["log_revenue"] = np.log10(df["revenue"].where(df["revenue"] > 0))

print("Tipos:")
display(df.dtypes)


## 1. Além do gênero, o que mais parece influenciar a nota que um filme recebe do público?

In [ ]:

# Correlação entre as principais variáveis numéricas e a nota
correlacoes = (
    df[["score", "budget_x", "revenue", "year"]]
    .corr(numeric_only=True)["score"]
    .drop("score")
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

display(correlacoes.to_frame("correlacao_com_nota"))

# Número de filmes por faixa de nota
df["faixa_nota"] = pd.cut(
    df["score"],
    bins=[-np.inf, 59.99, 69.99, 79.99, np.inf],
    labels=["< 60", "60–69", "70–79", "≥ 80"]
)

display(df["faixa_nota"].value_counts().sort_index())


In [ ]:

# Gênero: separar os gêneros que estão juntos em uma mesma célula
generos = (
    df[["genre", "score"]]
    .dropna(subset=["genre", "score"])
    .assign(genre=lambda x: x["genre"].str.split(r",\s*"))
    .explode("genre")
)

generos["genre"] = generos["genre"].str.replace("\xa0", " ", regex=False).str.strip()

resumo_genero = (
    generos.groupby("genre")["score"]
    .agg(media="mean", mediana="median", filmes="count")
    .query("filmes >= 100")
    .sort_values("media", ascending=False)
)

display(resumo_genero)

# Gráfico principal
graf_genero = resumo_genero.sort_values("media")

plt.figure(figsize=(9, 6))
plt.barh(graf_genero.index, graf_genero["media"])
plt.xlabel("Nota média IMDb")
plt.ylabel("Gênero")
plt.title("Nota média por gênero — mínimo de 100 filmes")
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()



### Conclusão — Pergunta 1

Além do gênero, as variáveis financeiras não mostram uma relação positiva forte com a nota. O orçamento apresenta correlação negativa com a avaliação, enquanto a receita apresenta relação linear muito fraca.

Isso sugere que **investir mais dinheiro não garante maior aceitação do público**. O gênero apresenta diferenças nas médias, mas também não deve ser interpretado como causa isolada do sucesso.

**Insight de negócio:** para aumentar a chance de aceitação, não é suficiente simplesmente aumentar o orçamento. Seria necessário investigar características de conteúdo, equipe criativa, direção, roteiro, franquia e posicionamento do filme — informações que o dataset atual não estrutura adequadamente.


## 2. Existe alguma coisa que esperávamos encontrar e não encontramos?

In [ ]:

# Inventário das colunas disponíveis
inventario = pd.DataFrame({
    "coluna": df.columns,
    "tipo": df.dtypes.astype(str).values,
    "nulos": df.isna().sum().values,
    "percentual_nulos": (df.isna().mean() * 100).round(2).values
})

display(inventario)



### Conclusão — Pergunta 2

O dataset possui título, data, nota, gênero, sinopse, `crew`, idioma, orçamento, receita e país. Porém, faltam informações estruturadas importantes para uma decisão de investimento:

- diretor;
- produtor;
- roteirista;
- estúdio/distribuidora;
- franquia/série;
- investimento em marketing;
- número de salas/alcance de distribuição;
- nota de críticos;
- bilheteria de abertura.

Se essas informações fossem necessárias, o próximo passo seria **enriquecer o dataset com outras fontes**, criando uma base integrada antes de tirar conclusões financeiras ou sobre profissionais.


## 3. Filmes muito bem avaliados pelo público costumam ter algo em comum?

In [ ]:

# Definindo "muito bem avaliados" como nota >= 80
muito_bem_avaliados = df[df["score"] >= 80].copy()
demais_filmes = df[df["score"] < 80].copy()

comparacao = pd.DataFrame({
    "grupo": ["Nota >= 80", "Nota < 80"],
    "quantidade": [len(muito_bem_avaliados), len(demais_filmes)],
    "orcamento_mediano": [
        muito_bem_avaliados["budget_x"].median(),
        demais_filmes["budget_x"].median()
    ],
    "receita_mediana": [
        muito_bem_avaliados["revenue"].median(),
        demais_filmes["revenue"].median()
    ],
    "nota_media": [
        muito_bem_avaliados["score"].mean(),
        demais_filmes["score"].mean()
    ]
})

display(comparacao)
print(f"Percentual de filmes com nota >= 80: {len(muito_bem_avaliados)/len(df)*100:.2f}%")



### Conclusão — Pergunta 3

Os filmes muito bem avaliados representam uma parcela pequena do dataset. Um ponto relevante é que eles **não apresentam orçamento mediano maior** que os demais filmes.

Isso reforça a hipótese de que qualidade percebida não depende apenas de recursos financeiros. O padrão de sucesso provavelmente envolve uma combinação de fatores criativos e de produção que não estão totalmente disponíveis no dataset.

**Importante:** o resultado indica associação, não prova quais fatores causam uma nota alta.


## 4. A informação sobre quem trabalhou em cada filme é suficiente?

In [ ]:

# Inspeção de alguns registros da coluna crew
display(df[["names", "crew"]].head(10))

# Quantidade média de itens listados em crew
df["crew_qtd_itens"] = (
    df["crew"]
    .fillna("")
    .apply(lambda x: len([item for item in x.split(",") if item.strip()]))
)

display(df["crew_qtd_itens"].describe())



### Conclusão — Pergunta 4

**Não.**

A coluna `crew` não apresenta uma estrutura que permita separar claramente diretor, produtor, roteirista e demais funções. Nos registros observados, aparecem nomes de pessoas associados a personagens, o que impede uma análise confiável sobre quem efetivamente construiu cada filme.

Portanto, essa variável pode ser explorada descritivamente, mas **não é suficiente para explicar o desempenho dos profissionais por trás dos filmes**.


## 5. Como descobrir um padrão de sucesso ligado às pessoas por trás dos filmes?


### Estratégia analítica proposta

Para responder corretamente a essa pergunta, eu faria um enriquecimento da base com:

`diretor`, `produtor`, `roteirista`, `estúdio` e `franquia`.

Depois, para cada profissional, calcularia:

- quantidade de filmes;
- nota média e mediana;
- percentual de filmes com nota ≥ 80;
- receita média e mediana;
- desempenho por gênero;
- desempenho por faixa de orçamento.

O ponto importante é **controlar o contexto**. Um diretor pode apresentar receita alta simplesmente porque trabalha com grandes franquias. Portanto, não basta comparar médias brutas.

A análise ideal seria uma combinação de agrupamentos, visualizações e, posteriormente, um modelo estatístico que controle gênero, orçamento, ano e franquia.


In [ ]:

# Diagnóstico da coluna crew:
# mostra os termos mais frequentes, sem tratá-los como "diretores".
from collections import Counter

nomes_crew = []
for valor in df["crew"].dropna():
    nomes_crew.extend([x.strip() for x in valor.split(",") if x.strip()])

frequencias_crew = (
    pd.Series(nomes_crew)
    .value_counts()
    .head(20)
    .rename("ocorrencias")
)

display(frequencias_crew.to_frame())


## 6. Os filmes que mais faturaram são também os mais bem avaliados? Sempre?

In [ ]:

# Top 15 filmes por receita
top_receita = (
    df[["names", "score", "budget_x", "revenue"]]
    .dropna(subset=["revenue"])
    .sort_values("revenue", ascending=False)
    .head(15)
)

display(top_receita)

# Percentual dos 10% maiores faturamentos que têm nota abaixo de 70
limite_top10 = df["revenue"].quantile(0.90)
top10_receita = df[df["revenue"] >= limite_top10].dropna(subset=["score"])

percentual_baixa_nota = (top10_receita["score"] < 70).mean() * 100

print(f"Limite dos 10% maiores faturamentos: {limite_top10:,.0f}")
print(f"Nos 10% maiores faturamentos, {percentual_baixa_nota:.1f}% têm nota abaixo de 70.")


In [ ]:

# Gráfico principal: receita x nota
dados_graf = df[["log_revenue", "score"]].dropna()

plt.figure(figsize=(9, 5.5))
plt.scatter(dados_graf["log_revenue"], dados_graf["score"], alpha=0.25, s=14)

coef = np.polyfit(dados_graf["log_revenue"], dados_graf["score"], 1)
x = np.linspace(dados_graf["log_revenue"].min(), dados_graf["log_revenue"].max(), 100)
plt.plot(x, np.polyval(coef, x), linewidth=2)

plt.xlabel("Receita (log10)")
plt.ylabel("Nota IMDb")
plt.title("Receita x Nota do público")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()



### Conclusão — Pergunta 6

**Não.**

Os maiores faturamentos não correspondem necessariamente às maiores notas. A relação linear entre receita e nota é fraca.

Isso é central para a decisão de negócio: **sucesso financeiro e aceitação do público devem ser tratados como KPIs diferentes**.

Um filme pode gerar muito dinheiro mesmo apresentando avaliação apenas mediana.


## 7. O que pode explicar um filme que fatura muito, mas recebe nota mediana ou baixa?


O dataset permite levantar hipóteses, mas não permite provar todas elas.

Possíveis explicações a investigar:

1. **Franquia conhecida:** o público já possui interesse antes do lançamento.
2. **Marketing:** campanhas grandes podem gerar demanda inicial.
3. **Distribuição:** lançamento em muitos mercados e salas amplia o potencial de receita.
4. **Personagens/universos conhecidos:** reduzem a necessidade de criar demanda do zero.
5. **Sequências:** um filme pode se beneficiar da audiência construída por títulos anteriores.
6. **Receita e satisfação medem fenômenos diferentes:** receita mede dinheiro gerado; nota mede avaliação do público.

Para testar essas hipóteses seria necessário adicionar variáveis de marketing, distribuição e franquia.


## 8. Filmes da mesma série/continuação se comportam de forma parecida?

In [ ]:

# O dataset não possui uma coluna estruturada de franquia.
# Vamos verificar se existe algum indício no título com palavras recorrentes,
# mas isso NÃO deve ser tratado como identificação oficial de franquias.

palavras_sequencia = [
    "Avengers", "Avatar", "Jurassic", "Star Wars", "Halloween",
    "King Kong", "Mission: Impossible", "Transformers"
]

for termo in palavras_sequencia:
    subset = df[df["names"].str.contains(termo, case=False, na=False)]
    if len(subset) > 1:
        print(f"\n{termo}")
        display(
            subset[["names", "score", "revenue"]]
            .sort_values("revenue", ascending=False)
        )



### Conclusão — Pergunta 8

O dataset **não permite uma resposta completa e sistemática**, porque não possui uma coluna identificando a franquia.

É possível encontrar alguns casos pelo nome dos filmes, mas isso seria uma classificação manual e incompleta.

Para responder corretamente, seria necessário criar/enriquecer uma variável `franchise_id` ou `franchise_name` e então comparar, dentro de cada franquia:

- evolução da nota;
- evolução da receita;
- orçamento;
- desempenho do primeiro filme versus continuações.

**Limitação importante:** não devemos afirmar que todas as sequências se comportam de maneira parecida ou diferente usando apenas a estrutura atual do dataset.


## 9. Quanto maior o orçamento, melhor a nota?

In [ ]:

# Correlação orçamento x nota
corr_budget = df[["budget_x", "score"]].corr().iloc[0, 1]

print(f"Correlação orçamento x nota: {corr_budget:.3f}")

# Comparação entre filmes muito bem avaliados e os demais
comparacao_orcamento = pd.DataFrame({
    "grupo": ["Nota >= 80", "Nota < 80"],
    "orcamento_mediano": [
        df.loc[df["score"] >= 80, "budget_x"].median(),
        df.loc[df["score"] < 80, "budget_x"].median()
    ]
})

display(comparacao_orcamento)


In [ ]:

# Gráfico principal: orçamento x nota
dados_graf = df[["log_budget", "score"]].dropna()

plt.figure(figsize=(9, 5.5))
plt.scatter(dados_graf["log_budget"], dados_graf["score"], alpha=0.25, s=14)

coef = np.polyfit(dados_graf["log_budget"], dados_graf["score"], 1)
x = np.linspace(dados_graf["log_budget"].min(), dados_graf["log_budget"].max(), 100)
plt.plot(x, np.polyval(coef, x), linewidth=2)

plt.xlabel("Orçamento (log10)")
plt.ylabel("Nota IMDb")
plt.title("Orçamento x Nota do público")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()



### Conclusão — Pergunta 9

Os dados **não confirmam** a hipótese de que um orçamento maior produz uma nota maior.

A correlação encontrada é negativa e fraca/moderada em magnitude. Além disso, os filmes com nota ≥ 80 apresentam orçamento mediano inferior ao dos demais filmes.

Isso não significa que orçamento seja irrelevante. O orçamento pode ser necessário para produção, elenco, efeitos, distribuição e marketing. O ponto é que **o orçamento, isoladamente, não explica a aceitação do público**.


## 10. Qualidade e confiabilidade dos dados

In [ ]:

# Valores faltantes
print("Valores ausentes por coluna:")
display(df.isna().sum().sort_values(ascending=False).to_frame("nulos"))

# Faixas suspeitas
print("\nNotas fora do intervalo esperado [0, 100]:")
display(df.loc[(df["score"] < 0) | (df["score"] > 100), ["names", "score"]])

print("\nQuantidade de notas exatamente 100:")
print((df["score"] == 100).sum())

print("\nOrçamento <= 0:")
print((df["budget_x"] <= 0).sum())

print("\nReceita <= 0:")
print((df["revenue"] <= 0).sum())

print("\nLinhas duplicadas completas:")
print(df.duplicated().sum())


In [ ]:

# Diagnóstico de repetição de pares financeiros
pares_financeiros = (
    df.groupby(["budget_x", "revenue"], dropna=False)
      .size()
      .reset_index(name="qtd_filmes")
      .query("qtd_filmes > 1")
      .sort_values("qtd_filmes", ascending=False)
)

print(f"Pares orçamento/receita repetidos: {len(pares_financeiros):,}")
print(f"Filmes pertencentes a pares repetidos: "
      f"{df.merge(pares_financeiros[['budget_x','revenue']],
                 on=['budget_x','revenue'], how='inner').shape[0]:,}")

display(pares_financeiros.head(15))



### Conclusão sobre qualidade dos dados

Foram encontrados sinais que recomendam cautela, especialmente nas variáveis financeiras:

- pares de orçamento/receita repetidos em muitos filmes;
- valores financeiros que merecem validação;
- notas extremas que precisam ser investigadas;
- ausência de variáveis importantes para explicar profissionais, franquias e marketing.

Por isso, as análises de receita e orçamento devem ser apresentadas como **exploratórias**, e não como base única para uma decisão real de investimento.



# Conclusão executiva

### O que os dados sugerem?

**1. Aceitação do público e retorno financeiro são dimensões diferentes.**  
A receita apresenta relação fraca com a nota.

**2. Mais orçamento não significa maior nota.**  
O orçamento não apresenta relação positiva com a avaliação do público.

**3. O gênero apresenta diferenças de nota média.**  
Entretanto, gênero sozinho não explica o sucesso.

**4. Filmes muito bem avaliados não dependem necessariamente de grandes orçamentos.**  
A mediana de orçamento dos filmes com nota ≥ 80 é menor que a dos demais filmes.

**5. O dataset não é suficiente para explicar o papel das pessoas por trás dos filmes.**  
Faltam informações estruturadas sobre direção, produção, roteiro e estúdios.

**6. Franquias também não podem ser analisadas de forma completa.**  
Falta uma identificação estruturada das séries/continuações.

**7. Há limitações de qualidade nos dados financeiros.**  
Antes de usar os resultados para investimento, orçamento e receita deveriam ser validados com uma fonte confiável.

### Recomendação de negócio

Em vez de procurar uma única variável que "garanta" sucesso, a próxima análise deveria construir um **modelo de sucesso multidimensional**, combinando:

`nota do público + receita + orçamento + gênero + diretor + produtor + estúdio + franquia + marketing + distribuição`.

O objetivo seria identificar perfis de filmes que apresentam simultaneamente **boa aceitação e bom retorno**, além dos casos em que existe **alto retorno com baixa reputação** ou **alta reputação com baixo retorno**.
